In [9]:
from pathlib import Path #this basically allows use to work with file and folder locations easily so that there is no version confusion or loss of where we are running from
import json # this lets us read and write json data
import re #regular expressions
from collections import Counter #counter is a dictionary subclass that counts how many times each word appears

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report #these are threee tools that help u scheck how good our models predictions will be

SEED = 42
torch.manual_seed(SEED)

In [10]:
# This is an alteration of the previous code "find_data_folder"
#The supplied hiddent test file is expected in assignment 2 titled "hidden_test_with_labels.csv" The code will check for this name.

def find_file(start_folder, candidate_names):
    likely_folders = [start_folder, start_folder / "assignment2", start_folder / "data"]
    for folder in likely_folders:
        for name in candidate_names:
            if (folder / name).exists():
                return folder / name
# if none of the likely folders work then we can fall back to a recursive search
    for name in candidate_names:
        for match in start_folder.rglob(name):
            return match
# if we get here then we were unable to find the data. Will reply a very clear error message.
    raise FileNotFoundError(
        f" Could not find any of {candidate_names}."
        "Place the hidden test CSV next to this notebook"
    )

ROOT = Path.cwd()
CHECKPOINT_DIR = ROOT / "model_checkpoint"
HIDDEN_TEST_PATH = find_file(ROOT, ["hidden_test.csv", "hidden_test_with_labels.csv"])
PUBLIC_TEST_PATH = find_file(ROOT, ["public_test.csv"])



In [11]:
# a regular expression 
TOKEN_RE = re.compile(r"[a-z']+")

def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

def encode(text, vocab):
    tokens = tokenize(text)
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
    if len(ids) == 0:
        ids = [vocab["<unk>"]]
    return ids

class ReviewDataset(Dataset):
    #renumbers rows 0,1,2 and so on. This is important because after a train/val split, the original pandas rows numbers are no longer consecutive
    def __init__(self, texts, labels, vocab):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True) if labels is not None else None
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

# lets pythons datset (idx) work. this is called once per row, per epoch
    def __getitem__(self, idx):
        ids = encode(self.texts.iloc[idx], self.vocab)
        label = int(self.labels.iloc[idx]) if self.labels is not None else -1
        return ids, label

#this pads each batch to the longest review in that batch with no fixed max length
def collate(batch):
    seqs, labels = zip(*batch)
    max_len = max(len(s) for s in seqs)
    x = torch.zeros(len(seqs), max_len, dtype=torch.long)
    for i, s in enumerate(seqs):
        x[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    y = torch.tensor(labels, dtype=torch.long)
    return x, y

class NeuralClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=150, hidden_dim=128, num_classes=2, pad_idx=0, dropout=0.6):
        super().__init__()
        # convert word IDs into vector representations
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        #first fully connected layer from embedding size to hidden size
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        # dropout layer to help prevent overfitting
        self.dropout = nn.Dropout(dropout)
        # our output layer hidden size to number of classes
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        #create a mask to mark non-padding tokens
        mask = (x != 0).unsqueeze(-1).float()
        # get word embedding and zero out any padding positions          # ignore <pad> tokens
        emb = self.embedding(x) * mask
        # Add up word embeddings across each sentence
        summed = emb.sum(dim=1)
        # count non padding tokens
        counts = mask.sum(dim=1).clamp(min=1)
        # get the average embedding for each sentence
        averaged = summed / counts
        hidden = torch.relu(self.fc1(averaged))
        hidden = self.dropout(hidden)
        return self.fc2(hidden)

def predict(model, vocab, texts, batch_size=64):
    # putting the model in evaluation mode this will turn off droput and will set up the data loader
    model.eval()
    ds = ReviewDataset(texts, None, vocab)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate)
    preds = []
    #disable gradient calculation to save memroy and speed up predictions
    with torch.no_grad():
        for xb, _ in dl:
            logits = model(xb) # here we get raw predictions and then pick the class with the highest score
            preds.extend(logits.argmax(dim=1).tolist())
#reutnr the final list of predicted class labels
    return preds

In [12]:
#inference only sanity check
with open(CHECKPOINT_DIR / "vocab.json", "r", encoding="utf-8") as f:
    vocab = json.load(f)
with open(CHECKPOINT_DIR / "metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

#recreate the model architecture using the loaded metadata
model = NeuralClassifier(
    vocab_size= metadata["vocab_size"],
    embed_dim= metadata["embed_dim"],
    hidden_dim= metadata["hidden_dim"],
    dropout= metadata["dropout"],
)


model.load_state_dict(torch.load(CHECKPOINT_DIR / "model_state_dict.pt"))
model.eval()

print("Loaded checkpoint trained on", metadata["train_rows"], "reviews")
print("Vocabulary size:", metadata["vocab_size"])
print("Stage 1 recorded CV accuracy:", metadata["cv_accuracy"])
print("Stage 1 recorded public test accuracy:", metadata["public_test_accuracy"])



Loaded checkpoint trained on 240 reviews
Vocabulary size: 8503
Stage 1 recorded CV accuracy: 0.6541666666666666
Stage 1 recorded public test accuracy: 0.5975


In [13]:
hidden_test = pd.read_csv(HIDDEN_TEST_PATH)
x_hidden = hidden_test["text"].fillna("")
y_hidden = hidden_test["label"].astype(int)

print("Hidden Test Reviews:", len(hidden_test))
print("Hidden Test Class Counts:")
print(y_hidden.value_counts().sort_index().rename(index={0: "Negative", 1: "Positive"}))

#generate predicitons
hidden_predictions = predict(model, vocab, x_hidden)
hidden_predictions = np.array(hidden_predictions).astype(int)

#calculate overall accuracy and build the confusion matrix
hidden_accuracy = accuracy_score(y_hidden, hidden_predictions)
hidden_matrix = confusion_matrix(y_hidden, hidden_predictions, labels=[0, 1])

#print total test accuracy and display the confusion matrix row by row
print(f"\nHidden Test Total Accuracy: {hidden_accuracy:.3f}")
print("Hidden Test Confusion Matrix (rows = true, columns = predicted, order = [negative, positive]):")
for row in hidden_matrix:
    print(row)

print("\n Hidden Test Classification Report:")
print(classification_report(y_hidden, hidden_predictions, target_names=["Negative", "Positive"]))

Hidden Test Reviews: 600
Hidden Test Class Counts:
label
Negative    300
Positive    300
Name: count, dtype: int64

Hidden Test Total Accuracy: 0.598
Hidden Test Confusion Matrix (rows = true, columns = predicted, order = [negative, positive]):
[156 144]
[ 97 203]

 Hidden Test Classification Report:
              precision    recall  f1-score   support

    Negative       0.62      0.52      0.56       300
    Positive       0.59      0.68      0.63       300

    accuracy                           0.60       600
   macro avg       0.60      0.60      0.60       600
weighted avg       0.60      0.60      0.60       600



In [14]:
public_test = pd.read_csv(PUBLIC_TEST_PATH)
x_public = public_test["text"].fillna("") # just in case there are empty rows
y_public = public_test["label"].astype(int)

# rerun predicitons on the public set using the loaded checkpoint
public_predictions = predict(model, vocab, x_public)
public_predictions = np.array(public_predictions).astype(int)

public_accuracy = accuracy_score(y_public, public_predictions)
public_matrix = confusion_matrix(y_public, public_predictions, labels=[0, 1])

print(f"Public Test Accuracy (recomputed from stage 1 checkpoint): {public_accuracy:.3f}")
print("Public Test Confusion Matrix")
for row in public_matrix:
    print(row)

# hidden test results, computed earlier
print(f"\nHidden Test Accuracy: {hidden_accuracy:.3f}")
print("Hidden Test Confusion Matrix:")
for row in hidden_matrix:
    print(row)


print(f"\nStage 1 recorded public test accuracy (from the metadata.json): {metadata["public_test_accuracy"]:.3f}")

# sanity check: make sure this is actually the same checkpoint from stage 1
assert abs(public_accuracy - metadata["public_test_accuracy"]) < 1e-6, (
    "Recomputed public test accuracy does not match the stage 1 recorded value,"
    "the checkpoint may not be the same one submitted in Stage 1."
)
print("Recomputed public test accuracy matches the stage 1 recorded value exactly, same checkpoint, no retraining.")

Public Test Accuracy (recomputed from stage 1 checkpoint): 0.598
Public Test Confusion Matrix
[108  92]
[ 69 131]

Hidden Test Accuracy: 0.598
Hidden Test Confusion Matrix:
[156 144]
[ 97 203]

Stage 1 recorded public test accuracy (from the metadata.json): 0.598
Recomputed public test accuracy matches the stage 1 recorded value exactly, same checkpoint, no retraining.


In [15]:
#build submission file with id and prediction label
hidden_submission = pd.DataFrame({
    "id": hidden_test["id"],
    "predicted_label": hidden_predictions,
})

hidden_submission.to_csv(ROOT / "hidden_test_predictions.csv", index=False)

print("Prediction file saved to:", ROOT / "hidden_test_predictions.csv")
hidden_submission.head() # quick peek to ensure that it looks right

Prediction file saved to: f:\CYSE 650 - Ai Tools and\HW\HW2\CYSE499_650_Summer2026-main\hidden_test_predictions.csv


,id,predicted_label
0,neg_cv795_10291,0
1,neg_cv174_9735,0
2,pos_cv065_15248,1
3,neg_cv076_26009,0
4,neg_cv417_14653,1


The fact that the accuracy managed to land around .60 on two separate independent tests just goes to shows that the ceiling here isnt luck or an unlucky test split, its the model itself. Its most likely being limited by learning all of its word embeddings from just 240 training reviews. If i had more time i would try.

- initializing the embedding layer with the pretrained word vectors instead of learning them from scratch, so the model starts from vectors that already encode word meaning instead of learnng that from 240 examples.

- fine tuning a smaller pretrained language model, wich the assignment explicitly allows and which would bring in far more general language than embeddings trained only on this dataset ever could.

- Agumenting the small imbalanced training set to grow the 60 review negative class specifically, instead of relying on class weighting alone



## Use of AI

Most the code used in this notebook was carried over from my stage 1 work, since stage 2 only required loading the existing checkpoint and running inference on the hidden test set. I used AI assitance in a few targeted places rather than for writing the core logic:

- Debugging a few of the runtime errors that halted execution when i first adapted the stage 1 code for this notebook. This included a few misspelled/mistyped variable names that were cuasing NameErrors and stopping the run partway through.
- Getting help tracking down why one cell wasn't executing correctly after the code was manually typed into this notebook
- Minor formatting and comment cleanup for readbility

No model architecture, training logic, or prediction code was written by AI. AI was used only for debugging and formatting assistance on top of my existing code. (claude was the code model used to help with these tasks)
